<!-- nav -->
[← 6 · Trees on the wire](06_Trees_On_The_Wire.ipynb) · [Home](../../README.md) · [8 · From constraints to code →](08_From_Constraints_To_Code.ipynb)

# 7 · From one language to another

**The problem.** A team's validation logic is TypeScript, in the browser; the batch job that must apply the same logic
nightly is Python. Porting by hand doubles the work and drifts; a transpiler that pastes strings breaks on the first
corner case.

A transpiler on trees maps one language's syntax nodes to another's, so it never pastes text, and it says exactly where
it can't follow. `TypeScriptToPython.transpile` covers the TypeScript that ordinary code is written in.

In [ ]:
import contextlib
import io

from mbse.Programs.Framework import Errors
from mbse.Programs.Python import Python314
from mbse.Programs.Transpilers.TypeScriptToPython import transpile
from mbse.Programs.TypeScript import TypeScript59

source = '''function label(n: number): string {
    if (n % 15 === 0) { return "FizzBuzz"; }
    if (n % 3 === 0) { return "Fizz"; }
    if (n % 5 === 0) { return "Buzz"; }
    return String(n);
}
for (let i = 1; i <= 15; i++) {
    console.log(label(i));
}
'''

## Step 1: translating

Parse the TypeScript, transpile the tree, print the Python:

In [ ]:
python = Python314.print(transpile(TypeScript59.parse(source)))
print(python)

Each construct maps to Python's idiom, not to a literal copy: `===` is `==`, `String(n)` is `str(n)`, `console.log`
is `print`, `number` is `float`, and a counted loop whose body doesn't assign its counter is a `range`.

## Step 2: it runs

The translation is a program, not a sketch:

In [ ]:
output = io.StringIO()
with contextlib.redirect_stdout(output):
    exec(python, {})
print(output.getvalue().split())

## Step 3: classes, getters and enums

Classes keep their shape: fields are assigned in `__init__`, a getter is a `@property`, an enum of numbers an
`IntEnum`:

In [ ]:
print(Python314.print(transpile(TypeScript59.parse('''enum Unit { Celsius, Fahrenheit }
class Reading {
    constructor(public value: number, public unit: Unit) {}
    get celsius(): number {
        return this.unit === Unit.Celsius ? this.value : (this.value - 32) * 5 / 9;
    }
}
console.log(new Reading(41, Unit.Fahrenheit).celsius);
'''))))

## Step 4: where it stops

What the transpiler doesn't cover, it refuses, naming the syntax node by its path, rather than guessing:

In [ ]:
try:
    transpile(TypeScript59.parse("outer: for (const row of rows) {\n    for (const x of row) { if (x) continue outer; }\n}\n"))
except Errors.TranspileError as error:
    print(error)

And what the two languages do differently, it doesn't emulate: `%` of a negative number, the truthiness of an empty
array, how numbers print. Translating keeps what code means where the languages agree; where they don't, the
difference is documented, and the tests compare what both programs print.

## Why it works this way

- **Tree to tree.** The transpiler reads a TypeScript tree and builds a Python tree with Python's builders; the
  Python printer handles precedence and layout. Nothing pastes text.
- **Names through definitions.** Whether `label` is a global to map or a function the program declares, whether a
  function assigns a variable of an enclosing scope (`nonlocal`), comes from `Definitions` (case study 4).
- **Refuse at a path.** A translation that silently approximated would be wrong in production. A `TranspileError` with
  a path says what to change.

<!-- nav -->
[← 6 · Trees on the wire](06_Trees_On_The_Wire.ipynb) · [Home](../../README.md) · [8 · From constraints to code →](08_From_Constraints_To_Code.ipynb)